In [1]:
!pip install -q voyageai faiss-cpu openai anthropic tqdm

In [3]:
import os
import json
import re
import fnmatch
import numpy as np
import faiss
import voyageai
from openai import OpenAI
from tqdm import tqdm
from google.colab import drive, userdata
import anthropic

drive.mount('/content/drive')


anthropic_client = anthropic.Anthropic(
    api_key=userdata.get('ANTHROPIC_API_KEY')
)
print("✓ anthropic_client initialized")

#try groq
from openai import OpenAI
from google.colab import userdata


gen_client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

# ── Paths ─────────────────────────────────────────────────────────────
DRIVE_BASE   = '/content/drive/MyDrive/iam_rag_project/'
FAISS_PATH   = DRIVE_BASE + 'indexes/dense/combined_faiss.index'
CHUNKS_PATH  = DRIVE_BASE + 'indexes/dense/combined_chunks.json'
GT_PATH      = DRIVE_BASE + 'data/evaluation/evaluation_ground_truth.json'
GT_DB_PATH   = DRIVE_BASE + 'data/processed/ground_truth_actions.json'
RESULTS_PATH = DRIVE_BASE + 'results/evaluation_results.json'

os.makedirs(DRIVE_BASE + 'results/', exist_ok=True)

# ── API clients ───────────────────────────────────────────────────────
openai_client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
voyage_client = voyageai.Client(api_key=userdata.get('VOYAGE_API_KEY'))

print("✓ Setup complete")

Mounted at /content/drive
✓ anthropic_client initialized
✓ Setup complete


In [4]:
class DenseRetriever:

    def __init__(self, chunks_path, faiss_index_path):
        # No voyage_api_key parameter — use global voyage_client instead
        print("Loading Dense Retriever...")
        with open(chunks_path) as f:
            self.chunks = json.load(f)
        self.index = faiss.read_index(faiss_index_path)
        print(f"  ✓ {len(self.chunks)} chunks, {self.index.ntotal} vectors")

    def embed_query(self, query):
        response = voyage_client.embed(  # ← global client
            texts=[query],
            model="voyage-code-3",
            input_type="query"
        )
        query_vec = np.array(
            [response.embeddings[0]], dtype='float32'
        )
        faiss.normalize_L2(query_vec)
        return query_vec

    def retrieve(self, query, k=30):
        query_vec = self.embed_query(query)
        distances, indices = self.index.search(query_vec, k)

        results = []
        for score, idx in zip(distances[0], indices[0]):
            if idx == -1:
                continue
            chunk = self.chunks[idx]
            results.append({
                'text':     chunk['text'],
                'metadata': chunk['metadata'],
                'score':    float(score)
            })
        return results


dense_retriever = DenseRetriever(CHUNKS_PATH, FAISS_PATH)

Loading Dense Retriever...
  ✓ 4978 chunks, 4978 vectors


In [5]:
# ── Utility and companion action rules ───────────────────────────────

def normalize_action_name(action):
    """Strip [permission only] and other suffixes from action names."""
    return action.split(' [')[0].strip()


COMPANION_RULES = {
    # ── EC2 resource creation ─────────────────────────────────────────
    "ec2:RunInstances":               ["ec2:CreateTags",
                                       "ec2:DescribeInstances",
                                       "ec2:DescribeInstanceStatus",
                                       "ec2:DescribeImages",
                                       "ec2:DescribeSubnets",
                                       "ec2:DescribeVpcs",
                                       "ec2:DescribeSecurityGroups"],
    "ec2:CreateNetworkInterface":     ["ec2:CreateTags",
                                       "ec2:DescribeSubnets",
                                       "ec2:DescribeVpcs",
                                       "ec2:DescribeNetworkInterfaces",
                                       "ec2:DescribeSecurityGroups"],
    "ec2:CreateSecurityGroup":        ["ec2:CreateTags",
                                       "ec2:DescribeVpcs",
                                       "ec2:DescribeSecurityGroups"],
    "ec2:DescribeInstances":          ["ec2:DescribeInstanceStatus",
                                       "ec2:DescribeImages"],
    "ec2:StartInstances":             ["ec2:StopInstances",
                                       "ec2:DescribeInstances",
                                       "ec2:DescribeInstanceStatus"],
    # ── ECS task operations ───────────────────────────────────────────
    "ecs:RunTask":                    ["ec2:CreateTags",
                                       "ec2:DescribeNetworkInterfaces",
                                       "ec2:DescribeSubnets",
                                       "ec2:DescribeSecurityGroups",
                                       "iam:PassRole"],
    "ecs:CreateService":              ["ec2:CreateTags",
                                       "ec2:DescribeSubnets",
                                       "ec2:DescribeVpcs",
                                       "ec2:DescribeSecurityGroups",
                                       "iam:PassRole"],
    "ecs:RegisterTaskDefinition":     ["iam:PassRole"],
    "ecs:UpdateService":              ["iam:PassRole"],
    # ── IAM role passing ──────────────────────────────────────────────
    "codepipeline:StartPipelineExecution": ["iam:PassRole"],
    "codepipeline:CreatePipeline":         ["iam:PassRole"],
    "cloudformation:CreateStack":          ["iam:PassRole",
                                            "iam:CreateRole"],
    "cloudformation:ExecuteChangeSet":     ["iam:PassRole"],
    "cloudformation:CreateChangeSet":      ["iam:PassRole"],
    "codebuild:StartBuild":                ["iam:PassRole"],
    "glue:StartJobRun":                    ["iam:PassRole"],
    "lambda:CreateFunction":               ["iam:PassRole"],
    "states:StartExecution":               ["iam:PassRole"],
    # ── CloudWatch Logs ───────────────────────────────────────────────
    "logs:CreateLogStream":           ["logs:CreateLogGroup",
                                       "logs:PutLogEvents"],
    "logs:PutLogEvents":              ["logs:CreateLogGroup",
                                       "logs:CreateLogStream"],
    "logs:CreateLogGroup":            ["logs:CreateLogStream",
                                       "logs:PutLogEvents"],
    # ── ECR image pulling ─────────────────────────────────────────────
    "ecr:BatchGetImage":              ["ecr:GetAuthorizationToken",
                                       "ecr:BatchCheckLayerAvailability",
                                       "ecr:GetDownloadUrlForLayer"],
    "ecr:GetDownloadUrlForLayer":     ["ecr:GetAuthorizationToken",
                                       "ecr:BatchGetImage",
                                       "ecr:BatchCheckLayerAvailability"],
    "ecr:BatchCheckLayerAvailability":["ecr:GetAuthorizationToken",
                                       "ecr:BatchGetImage",
                                       "ecr:GetDownloadUrlForLayer"],
    # ── ECR image pushing ─────────────────────────────────────────────
    "ecr:PutImage":                   ["ecr:GetAuthorizationToken",
                                       "ecr:InitiateLayerUpload",
                                       "ecr:UploadLayerPart",
                                       "ecr:CompleteLayerUpload",
                                       "ecr:BatchCheckLayerAvailability"],
    "ecr:InitiateLayerUpload":        ["ecr:GetAuthorizationToken",
                                       "ecr:UploadLayerPart",
                                       "ecr:CompleteLayerUpload",
                                       "ecr:PutImage"],
    # ── DynamoDB streams ──────────────────────────────────────────────
    "dynamodb:GetRecords":            ["dynamodb:GetShardIterator",
                                       "dynamodb:DescribeStream",
                                       "dynamodb:ListStreams"],
    "dynamodb:GetShardIterator":      ["dynamodb:GetRecords",
                                       "dynamodb:DescribeStream",
                                       "dynamodb:ListStreams"],
    # ── SQS consumer ─────────────────────────────────────────────────
    "sqs:ReceiveMessage":             ["sqs:DeleteMessage",
                                       "sqs:GetQueueAttributes"],
    "sqs:DeleteMessage":              ["sqs:ReceiveMessage",
                                       "sqs:GetQueueAttributes"],
    # ── Kinesis stream reading ────────────────────────────────────────
    "kinesis:GetRecords":             ["kinesis:GetShardIterator",
                                       "kinesis:DescribeStream",
                                       "kinesis:ListStreams",
                                       "kinesis:DescribeStreamSummary"],
    "kinesis:GetShardIterator":       ["kinesis:GetRecords",
                                       "kinesis:DescribeStream",
                                       "kinesis:ListShards"],
    "kinesis:PutRecord":              ["kinesis:PutRecords",
                                       "kinesis:DescribeStream"],
    # ── S3 bucket access ──────────────────────────────────────────────
    "s3:GetObject":                   ["s3:ListBucket"],
    "s3:PutObject":                   ["s3:GetObject", "s3:ListBucket"],
    "s3:ListBucket":                  ["s3:GetObject",
                                       "s3:GetBucketLocation"],
    # ── KMS ──────────────────────────────────────────────────────────
    "kms:Decrypt":                    ["kms:DescribeKey",
                                       "kms:GenerateDataKey"],
    "kms:GenerateDataKey":            ["kms:Decrypt", "kms:DescribeKey"],
    "kms:Encrypt":                    ["kms:DescribeKey",
                                       "kms:GenerateDataKey"],
    # ── Secrets Manager ───────────────────────────────────────────────
    "secretsmanager:GetSecretValue":  ["secretsmanager:DescribeSecret",
                                       "kms:Decrypt"],
    # ── CodeDeploy ────────────────────────────────────────────────────
    "codedeploy:CreateDeployment":    ["codedeploy:GetDeployment",
                                       "codedeploy:GetDeploymentConfig",
                                       "codedeploy:GetDeploymentGroup"],
    "codedeploy:GetDeployment":       ["codedeploy:GetDeploymentConfig",
                                       "codedeploy:ListDeployments"],
    # ── Step Functions ────────────────────────────────────────────────
    "states:DescribeExecution":       ["states:GetExecutionHistory",
                                       "states:ListExecutions"],
    # ── RDS ───────────────────────────────────────────────────────────
    "rds:CreateDBInstance":           ["rds:DescribeDBInstances",
                                       "rds:DescribeDBSubnetGroups",
                                       "ec2:DescribeVpcs",
                                       "ec2:DescribeSubnets",
                                       "ec2:DescribeSecurityGroups"],
    "rds:ModifyDBInstance":           ["rds:DescribeDBInstances"],
    # ── Lambda ────────────────────────────────────────────────────────
    "lambda:CreateEventSourceMapping":["lambda:ListEventSourceMappings",
                                       "lambda:GetEventSourceMapping"],
    # ── CloudWatch metrics ────────────────────────────────────────────
    "cloudwatch:PutMetricData":       ["cloudwatch:GetMetricStatistics",
                                       "cloudwatch:ListMetrics"],
    "cloudwatch:PutMetricAlarm":      ["cloudwatch:DescribeAlarms",
                                       "cloudwatch:GetMetricData"],
}


def apply_companion_rules(retrieved_chunks, dense_retriever):
    """
    Post-retrieval augmentation using deterministic companion rules.
    Appends companion action chunks for any trigger actions retrieved.
    Uses consistent lowercase normalization to avoid case mismatch bugs.
    """
    # Collect retrieved actions normalized to lowercase
    retrieved_actions = set()
    for chunk in retrieved_chunks:
        if chunk["metadata"]["type"] == "iam_action":
            action = normalize_action_name(
                chunk["metadata"].get("action", "")
            ).lower()
            retrieved_actions.add(action)

    # Build lowercase version of COMPANION_RULES
    companion_rules_lower = {
        k.lower(): [v.lower() for v in vals]
        for k, vals in COMPANION_RULES.items()
    }

    # Find companions needed
    companions_to_add = set()
    for action in retrieved_actions:
        if action in companion_rules_lower:
            for companion in companion_rules_lower[action]:
                if companion not in retrieved_actions:
                    companions_to_add.add(companion)

    if not companions_to_add:
        return retrieved_chunks

    # Look up companion chunks from corpus
    companion_chunks = []
    for chunk in dense_retriever.chunks:
        if chunk["metadata"]["type"] != "iam_action":
            continue
        action = normalize_action_name(
            chunk["metadata"].get("action", "")
        ).lower()
        if action in companions_to_add:
            companion_chunk = dict(chunk)
            companion_chunk["source"] = "companion_rule"
            companion_chunk["score"]  = 1.0
            companion_chunks.append(companion_chunk)
            companions_to_add.discard(action)
        if not companions_to_add:
            break

    return retrieved_chunks + companion_chunks


print(f"✓ normalize_action_name() defined")
print(f"✓ COMPANION_RULES defined — {len(COMPANION_RULES)} trigger actions")
print(f"✓ apply_companion_rules() defined")

✓ normalize_action_name() defined
✓ COMPANION_RULES defined — 48 trigger actions
✓ apply_companion_rules() defined


In [6]:
#Decomposer + retrieval function

def decompose_query_gpt4o(query):
    prompt = f"""You are an AWS IAM expert with deep knowledge
of AWS service dependencies and implicit permission requirements.

Given this AWS task:
"{query}"

Your job is to decompose it into HIGH-PRECISION retrieval sub-queries
that recover BOTH primary actions and companion actions commonly required
for a working IAM policy. Identify ALL AWS services and permission types needed,
including IMPLICIT requirements that are not mentioned
in the task but are always needed.

Please follow these rules when identifying services:
1. Maximum 4 sub-queries total (explicit + implicit combined)
2. NO duplicate services — each service appears at most ONCE
3. Sub-query phrases must be ACTION-BUNDLE oriented and specific enough
   to retrieve exact IAM actions.
   BAD:  "KMS decrypt"
   GOOD: "KMS decrypt describe key"
   BAD:  "DynamoDB read stream"
   GOOD: "DynamoDB stream get records get shard iterator describe stream list streams"

4. Only add implicit services if they are UNIVERSALLY required:
   - Lambda ALWAYS needs: "CloudWatch Logs create log group stream put events"
   - ECR pull ALWAYS needs: "ECR get authorization token"
   DO NOT add KMS, IAM, ELB unless explicitly mentioned in task

5. Implicit service rules — ONLY add these when the named trigger is present:
   - 'logs' sub-query: ONLY if task mentions Lambda, ECS, or API Gateway
   - 'ecr' sub-query:  ONLY if task mentions pulling/running container images
   - 'kms' sub-query:  ONLY if task explicitly mentions encryption or KMS
   - 'sts' sub-query:  ONLY if task mentions assuming a role or cross-account
   DO NOT add implicit services for S3-only, SNS-only, or SQS-only tasks.

6. For services that have both a TRIGGERING/PUBLISH action and a
   CONFIGURATION/PERMISSION action, include BOTH in the query phrase:
   GOOD: "EventBridge put events put rule put targets configure"
   GOOD: "Lambda invoke function add permission event source"

7. Sub-query text must be specific enough to rank exact action chunks
   above generic ones. Include the exact action verb names in the phrase.

8. Companion action rules — when you identify these primary services
   or actions in a sub-query, ALWAYS include their companion terms
   in the SAME sub-query phrase so the retriever can find them:
   - EC2 instance or resource creation or management:
     → always add: "create tags describe subnets describe vpcs
       describe security groups describe network interfaces"
   - Any service that passes execution to another service
     (CodePipeline, CloudFormation, ECS, Glue, Lambda creation):
     → always add: "pass role iam passrole"
   - CloudWatch Logs writing:
     → always include all three: "create log group create log stream put log events"
   - ECR image pulling:
     → always add: "get authorization token"
   - SQS message consumption:
     → always add: "delete message get queue attributes"
   - Kinesis stream reading:
     → always add: "get shard iterator describe stream list streams"
   - KMS encryption or decryption:
     → always add: "describe key"
   - Secrets Manager access:
     → always add: "describe secret"
   - SQL query execution or data querying:
      → likely athena: start query execution get query execution
        get named query — NOT rds unless task says relational database
   These companion terms must appear in the sub-query TEXT,
   not as separate sub-queries.

Return a JSON object:
{{
  "explicit_services": [
    {{
      "service": "aws_service_prefix",
      "query": "specific action-level search phrase",
      "reason": "needs action1, action2, action3 for this task"
    }}
  ],
  "implicit_services": [
    {{
      "service": "aws_service_prefix",
      "query": "specific action-level search phrase",
      "reason": "always needed: action1, action2, action3"
    }}
  ]
}}

Return ONLY valid JSON."""

    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=500
    )
    try:
        text = response.choices[0].message.content.strip()
        if '```' in text:
            text = text.split('```')[1].split('```')[0]
            if text.startswith('json'):
                text = text[4:]
        result = json.loads(text.strip())
        sub_queries = []
        for item in result.get('explicit_services', []):
            sub_queries.append({
                'query':   item['query'],
                'type':    'explicit',
                'service': item['service'],
                'reason':  item.get('reason', '')
            })
        for item in result.get('implicit_services', []):
            sub_queries.append({
                'query':   item['query'],
                'type':    'implicit',
                'service': item['service'],
                'reason':  item.get('reason', '')
            })
        return sub_queries
    except Exception as e:
        print(f"  Decomposition failed: {e}")
        return [{'query': query, 'type': 'original', 'service': 'unknown', 'reason': ''}]

def retrieve_for_generation_decomposed_v3(query,
                                           k_per_subquery=3,
                                           k_policies=2,
                                          score_threshold=0.55):
    """
    Decomposed retrieval using GPT-4o as decomposer.
    GPT-3.5 remains the generator — only decomposition upgraded.
    """
    sub_queries = decompose_query_gpt4o(query)

    print(f"  GPT-4o decomposed into {len(sub_queries)} sub-queries:")
    for sq in sub_queries:
        tag = f"[{sq['type']}]"
        print(f"    {tag:<12} {sq['service']}: {sq['query']}")
        print(f"               reason: {sq['reason'][:60]}")

    seen_actions = set()
    all_actions  = []

    for sub_query_info in sub_queries:
        sub_query = sub_query_info['query']
        results   = dense_retriever.retrieve(sub_query, k=30)
        actions   = [r for r in results
                     if r['metadata']['type'] == 'iam_action']

        added = 0
        for r in actions:
            action = r['metadata']['action']
            clean  = action.split(' [')[0]

            if r['score'] < score_threshold:
                break
            if clean not in seen_actions and added < k_per_subquery:
                seen_actions.add(clean)
                r['source']    = f"action_{sub_query_info['type']}"
                r['sub_query'] = sub_query
                all_actions.append(r)
                added += 1

    # NEW — one policy per sub-query, deduplicated
    seen_policies = set()
    all_policies  = []

    for sub_query_info in sub_queries:
        sub_query   = sub_query_info['query']
        results_sq  = dense_retriever.retrieve(sub_query, k=30)
        policies_sq = [r for r in results_sq
                      if r['metadata']['type'] == 'iam_policy']

        for r in policies_sq:
            name = r['metadata'].get('policy_name', '')
            n_actions = len(r['metadata'].get('actions_used', []))
            if name not in seen_policies and n_actions <= 15:
                seen_policies.add(name)
                r['source'] = 'policy_decomposed'
                all_policies.append(r)
                break

    results = all_actions.copy()
    for r in all_policies[:k_policies]:
        results.append(r)

    print(f"  Total retrieved: {len(results)} chunks "
          f"({len(all_actions)} actions + "
          f"{len(results)-len(all_actions)} policies)")

    results = apply_companion_rules(results, dense_retriever)
    return results

In [7]:
# ── Generation functions ─────────────────────────
def generate_policy_no_rag(query):
    """Generate IAM policy with NO retrieval context."""
    prompt = f"""You are an AWS IAM expert.
Generate a valid IAM policy JSON document for this task:

"{query}"

Requirements:
- Return ONLY valid JSON, no explanation
- Follow exact IAM policy JSON structure
- Use only real AWS IAM action names
- Include Version and Statement fields

IAM Policy:"""

    response = gen_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=2000
    )
    return response.choices[0].message.content.strip()


def generate_policy_with_rag(query, retrieved_chunks):
    """Generate IAM policy WITH retrieved context."""

    # ── Pass 1: services directly named in retrieved action chunks ──────
    direct_services = {
        r['metadata']['action'].split(':')[0]
        for r in retrieved_chunks
        if r['metadata']['type'] == 'iam_action'
    }

    # ── Pass 1 policy scan: collect ALL actions from relevant policies ──
    # We need a first pass to discover dependent services (e.g. a policy
    # for an ECS task also contains ecr:* and logs:* actions — those
    # service prefixes must be allowed through in the final filter).
    dependent_services = set()
    for r in retrieved_chunks:
        if r['metadata']['type'] != 'iam_policy':
            continue
        doc = r['metadata']['policy_document']
        for stmt in doc.get('Statement', []):
            acts = stmt.get('Action', [])
            if isinstance(acts, str):
                acts = [acts]
            for a in acts:
                svc = a.split(':')[0]
                # Only expand from actions whose service is already relevant
                # — prevents completely unrelated services bleeding in
                if svc in direct_services or '*' in a:
                    dependent_services.add(svc)
                    # Also capture the co-occurring services in this stmt
                    for a2 in (acts if isinstance(acts, list) else [acts]):
                        dependent_services.add(a2.split(':')[0])

    relevant_services = direct_services | dependent_services

    # ── Action context: name + description + resource types ────────────
    action_lines = []
    for r in retrieved_chunks:
        if r['metadata']['type'] != 'iam_action':
            continue
        action         = r['metadata']['action']
        description    = r['metadata'].get('description', '')[:80]
        resource_types = r['metadata'].get('resource_types', [])
        resource_str   = (
            '  [resources: ' + ', '.join(resource_types) + ']'
            if resource_types else ''
        )
        action_lines.append('- ' + action + ': ' + description + resource_str)

    action_context = '\n'.join(action_lines)

    # ── Policy context: filtered to relevant + dependent services ───────
    policy_sections = []
    for r in retrieved_chunks:
        if r['metadata']['type'] != 'iam_policy':
            continue
        doc = r['metadata']['policy_document']

        filtered_actions = []
        for stmt in doc.get('Statement', []):
            acts = stmt.get('Action', [])
            if isinstance(acts, str):
                acts = [acts]
            for a in acts:
                service = a.split(':')[0]
                if (not relevant_services
                        or '*' in a
                        or service in relevant_services):
                    filtered_actions.append(a)

        if not filtered_actions:
            continue

        section = (
            'Policy name: ' + r['metadata']['policy_name'] + '\n'
            'Relevant actions from this policy:\n'
            + '\n'.join('  - ' + a for a in filtered_actions)
        )
        policy_sections.append(section)

    policy_context = '\n\n'.join(policy_sections)

    # ── Build prompt ────────────────────────────────────────────────────
    prompt = (
        'You are an AWS IAM expert.\n'
        'Generate a valid IAM policy JSON for this task:\n\n'
        '"' + query + '"\n\n'
        'INSTRUCTIONS — follow these steps in order:\n\n'
        'STEP 1: Read the example policies below carefully.\n'
        'The actions listed are REAL verified AWS actions.\n'
        'Include ALL actions from these policies that are\n'
        'relevant to the task above.\n\n'
        '<verified_policies>\n'
        + policy_context +
        '\n</verified_policies>\n\n'
        'STEP 2: The following verified actions are also relevant\n'
        'to this task — include all of them:\n\n'
        '<verified_actions>\n'
        + action_context +
        '\n</verified_actions>\n\n'
        'STEP 3: Review the actions you have collected from STEP 1 and STEP 2.\n'
        'Remove any action that is NOT directly required for the specific\n'
        'task described above. Keep only actions the task cannot function without.\n\n'
        'Specific removal rules:\n'
        '  - Remove s3:GetBucketLocation unless the task requires\n'
        '    determining bucket region explicitly\n'
        '  - Remove kms:Decrypt unless the task explicitly mentions\n'
        '    reading encrypted data\n'
        '  - Remove lambda:AddPermission and lambda:CreateEventSourceMapping\n'
        '    unless the task explicitly mentions configuring event sources\n'
        '  - Remove logging actions (logs:*) unless the service writes\n'
        '    execution logs (Lambda, ECS, CodeBuild, API Gateway)\n'
        '  - Remove companion EC2 describe actions unless the task\n'
        '    involves creating or modifying EC2 network resources\n\n'
        'IMPORTANT:\n'
        '- Return ONLY the filtered minimal action set\n'
        '- Prefer under-permission over over-permission\n'
        '- Return ONLY valid JSON with Version and Statement fields\n\n'
        'IAM Policy:'
    )

    try:
        import json as json_test
        json_test.dumps({'role': 'user', 'content': prompt})
    except Exception as e:
        print(f'Prompt serialization error: {e}')
        prompt = prompt.encode('utf-8', errors='ignore').decode('utf-8')

    response = gen_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0,
        max_tokens=2000
    )
    return response.choices[0].message.content.strip()


print('✓ generate_policy_with_rag() updated — expanded service filtering')

✓ generate_policy_with_rag() updated — expanded service filtering


In [8]:
#Four-layer evaluation functions

# Load ground truth action database for Layer 2
with open(GT_DB_PATH) as f:
    gt_database = json.load(f)

# Build normalized valid action set once
valid_actions_normalized = {
    normalize_action_name(a).lower()
    for a in gt_database['valid_actions']
}
print(f"✓ GT database loaded: {len(valid_actions_normalized)} valid actions")


# ── Shared extraction utility ─────────────────────────────────────────
def extract_actions_from_policy(policy_doc):
    """Extract all Allow actions as normalized lowercase set."""
    actions = set()
    for stmt in policy_doc.get("Statement", []):
        if stmt.get("Effect") != "Allow":
            continue
        raw = stmt.get("Action", [])
        if isinstance(raw, str):
            raw = [raw]
        for a in raw:
            actions.add(normalize_action_name(a).lower())
    return actions


def parse_policy_string(policy_str):
    """Parse policy JSON string, handling markdown code blocks."""
    try:
        clean = policy_str.strip()
        if '```json' in clean:
            clean = clean.split('```json')[1].split('```')[0]
        elif '```' in clean:
            clean = clean.split('```')[1].split('```')[0]
        return json.loads(clean.strip()), None
    except json.JSONDecodeError as e:
        return None, str(e)


# ── Layer 1: Syntactic validity ───────────────────────────────────────
def check_layer1_syntactic(policy_str):
    """
    Check if generated output is valid JSON with correct IAM structure.
    Returns (is_valid, message, policy_doc_or_None)
    """
    policy_doc, error = parse_policy_string(policy_str)

    if policy_doc is None:
        return False, f"Invalid JSON: {error}", None

    if 'Statement' not in policy_doc:
        return False, "Missing Statement field", None

    stmts = policy_doc['Statement']
    if not isinstance(stmts, list):
        return False, "Statement must be a list", None

    for i, stmt in enumerate(stmts):
        if not isinstance(stmt, dict):
            return False, f"Statement[{i}] is not a dict", None
        if 'Effect' not in stmt:
            return False, f"Statement[{i}] missing Effect", None
        if stmt['Effect'] not in ['Allow', 'Deny']:
            return False, f"Statement[{i}] invalid Effect: {stmt['Effect']}", None
        if 'Action' not in stmt:
            return False, f"Statement[{i}] missing Action", None
        if 'Resource' not in stmt:
            return False, f"Statement[{i}] missing Resource", None

    return True, "Valid", policy_doc


# ── Layer 2: Hallucination check ──────────────────────────────────────
def check_layer2_hallucination(policy_doc):
    """
    Check which generated actions don't exist in AWS.
    Handles [permission only] suffixes and wildcards.
    """
    generated_actions = extract_actions_from_policy(policy_doc)
    hallucinated = []
    valid        = []

    for action in generated_actions:
        clean = action.lower()

        # Direct match
        if clean in valid_actions_normalized:
            valid.append(action)
            continue

        # Wildcard — check if it matches any valid action
        if '*' in clean:
            matches = [
                v for v in valid_actions_normalized
                if fnmatch.fnmatch(v, clean)
            ]
            if matches:
                valid.append(action)
                continue

        hallucinated.append(action)

    n_gen  = len(generated_actions)
    n_hall = len(hallucinated)

    return {
        "n_generated":        n_gen,
        "n_hallucinated":     n_hall,
        "hallucination_rate": round(n_hall / n_gen, 3) if n_gen else 0,
        "hallucinated":       sorted(hallucinated),
    }


# ── Layer 3: Semantic action matching + resource + condition ──────────
def get_arn_service(arn):
    """Extract service from ARN. Returns * for wildcard."""
    if not arn or arn == "*" or not str(arn).startswith("arn:"):
        return "*"
    parts = str(arn).split(":")
    return parts[2] if len(parts) > 2 else "*"


def normalize_arn(arn):
    """Replace CloudFormation params and specific IDs with wildcards."""
    if not arn or arn == "*":
        return "*"
    arn = str(arn)
    arn = re.sub(r'\$\{[^}]+\}', '*', arn)      # ${AWS::Region} → *
    arn = re.sub(r':\d{12}:', ':*:', arn)         # account ID → *
    arn = re.sub(                                  # region → *
        r':(us|eu|ap|sa|ca|me|af)-[a-z]+-\d+:',
        ':*:', arn
    )
    return arn


def build_action_resource_map(policy_doc):
    """Map each action to its resource. Wildcard wins if seen multiple times."""
    action_resource = {}
    action_condition = {}

    for stmt in policy_doc.get("Statement", []):
        if stmt.get("Effect") != "Allow":
            continue
        resource  = stmt.get("Resource", "*")
        condition = stmt.get("Condition", {})
        actions   = stmt.get("Action", [])
        if isinstance(actions, str):
            actions = [actions]

        for a in actions:
            clean = normalize_action_name(a).lower()
            if action_resource.get(clean) == "*":
                continue  # already most permissive
            action_resource[clean]  = resource
            if condition:
                action_condition[clean] = condition

    return action_resource, action_condition


def check_resource_compatible(gen_res, gt_res):
    """
    Check resource compatibility. Returns (compatible, reason).
    GT wildcard → always accept.
    GT specific ARN → check service namespace matches.
    Generator wildcard vs specific GT → acceptable, flag for LLM judge.
    """
    gt_norm  = normalize_arn(gt_res)
    gen_norm = normalize_arn(gen_res if gen_res else "*")

    if gt_norm == "*":
        return True, "gt_wildcard"
    if gen_norm == "*":
        return True, "gen_wildcard_gt_specific"

    gt_svc  = get_arn_service(gt_norm)
    gen_svc = get_arn_service(gen_norm)

    if gt_svc == "*" or gen_svc == "*":
        return True, "wildcard_service"
    if gt_svc == gen_svc:
        return True, "service_match"

    return False, f"service_mismatch: gt={gt_svc} gen={gen_svc}"


def check_layer3_semantic(generated_policy_doc, gt_policy_doc):
    """
    Layer 3: Action set precision/recall/F1 with wildcard expansion
    + resource compatibility check (bonus metric)
    + condition presence check
    """
    gen_actions = extract_actions_from_policy(generated_policy_doc)
    gt_actions  = extract_actions_from_policy(gt_policy_doc)

    gen_res_map, gen_cond_map = build_action_resource_map(generated_policy_doc)
    gt_res_map,  gt_cond_map  = build_action_resource_map(gt_policy_doc)

    # ── Wildcard expansion ────────────────────────────────────────────
    covered_gt = set()
    extra_gen  = set()

    for gen_action in gen_actions:
        if '*' in gen_action:
            matched = {
                gt for gt in gt_actions
                if fnmatch.fnmatch(gt, gen_action)
            }
            if matched:
                covered_gt.update(matched)
            else:
                extra_gen.add(gen_action)
        else:
            if gen_action in gt_actions:
                covered_gt.add(gen_action)
            else:
                extra_gen.add(gen_action)

    missing_gt = gt_actions - covered_gt

    # ── Action metrics ────────────────────────────────────────────────
    precision = len(covered_gt) / len(gen_actions) if gen_actions else 0
    recall    = len(covered_gt) / len(gt_actions)  if gt_actions  else 0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0)

    # ── Resource compatibility (bonus — only for covered actions) ─────
    resource_checks     = []
    resource_mismatches = []

    for action in covered_gt:
        gt_res  = gt_res_map.get(action, "*")
        gen_res = gen_res_map.get(action, "*")
        compatible, reason = check_resource_compatible(gen_res, gt_res)
        resource_checks.append(compatible)
        if not compatible:
            resource_mismatches.append({
                "action":      action,
                "gt_resource": gt_res,
                "gen_resource": gen_res,
                "reason":      reason
            })

    resource_score = (
        sum(resource_checks) / len(resource_checks)
        if resource_checks else 1.0
    )

    # ── Condition presence check ──────────────────────────────────────
    gt_has_conditions  = bool(gt_cond_map)
    gen_has_conditions = bool(gen_cond_map)

    # Flag if GT has conditions but generator omitted them entirely
    condition_present_when_expected = (
        not gt_has_conditions or gen_has_conditions
    )

    # ── Deny statement detection ──────────────────────────────────────
    has_deny = any(
        stmt.get("Effect") == "Deny"
        for stmt in generated_policy_doc.get("Statement", [])
    )

    return {
        # Core action metrics
        "precision":    round(precision, 3),
        "recall":       round(recall, 3),
        "f1":           round(f1, 3),
        "exact_match":  (covered_gt == gt_actions and not extra_gen),
        "covered":      sorted(covered_gt),
        "missing":      sorted(missing_gt),
        "extra":        sorted(extra_gen),
        "n_generated":  len(gen_actions),
        "n_expected":   len(gt_actions),
        # Resource (bonus)
        "resource_score":       round(resource_score, 3),
        "resource_mismatches":  resource_mismatches,
        # Condition presence
        "gt_has_conditions":              gt_has_conditions,
        "gen_has_conditions":             gen_has_conditions,
        "condition_present_when_expected": condition_present_when_expected,
        # Structure
        "has_deny": has_deny,
    }


def check_layer4_llm_judge(query, generated_policy_doc,
                            gt_policy_doc, layer3_results):
    """
    LLM-as-judge using Claude: evaluates semantic correctness
    beyond action matching.
    """
    gen_str  = json.dumps(generated_policy_doc, indent=2)
    gt_str   = json.dumps(gt_policy_doc,        indent=2)
    missing  = layer3_results.get("missing", [])
    extra    = layer3_results.get("extra", [])
    gt_conds = layer3_results.get("gt_has_conditions", False)

    condition_instruction = ""
    if gt_conds:
      # Extract GT and generated conditions for explicit display
      gt_conditions = {
          i: stmt.get("Condition")
          for i, stmt in enumerate(gt_policy_doc.get("Statement", []))
          if stmt.get("Condition")
      }
      gen_conditions = {
          i: stmt.get("Condition")
          for i, stmt in enumerate(generated_policy_doc.get("Statement", []))
          if stmt.get("Condition")
      }

      condition_instruction = f"""
  The ground truth policy includes condition blocks:
  {json.dumps(gt_conditions, indent=2)}

  Generated policy conditions (if any):
  {json.dumps(gen_conditions, indent=2) if gen_conditions else "None — generator omitted conditions entirely"}

  Additionally evaluate:
  - Whether the generated policy includes appropriate conditions for this task
  - Whether omitting conditions (if generator did) creates a security risk
    for this specific task
  - Whether the condition keys and values used are semantically appropriate
    even if not identical to ground truth
  """

    prompt = f"""You are an AWS IAM security expert evaluating a generated IAM policy.

Task description: {query}

Ground truth policy:
{gt_str}

Generated policy:
{gen_str}

Objective analysis:
- Missing actions: {missing}
- Extra actions beyond expected: {extra}
- Resource compatibility score: {layer3_results.get('resource_score', 1.0)}
{condition_instruction}

Evaluate the generated policy on these dimensions:

1. FUNCTIONAL CORRECTNESS (0-10): Does the policy grant sufficient permissions
   for the task? Penalize for missing actions that would prevent the task.

2. SECURITY CORRECTNESS (0-10): Does the policy follow least-privilege?
   Penalize for unnecessary broad permissions or wildcard actions where
   specific actions could be used.

3. RESOURCE SCOPING (0-10): Are resources appropriately scoped?
   Using * is acceptable when the task does not specify resources.
   Penalize only if resources are clearly wrong service namespace.

4. OVERALL ACCEPTABILITY: Would an AWS security engineer approve this policy
   for the described task? Answer true or false.

5. KEY ISSUES: List the 1-3 most important problems if any.

Respond in this exact JSON format with no other text:
{{
  "functional_score":  <0-10>,
  "security_score":    <0-10>,
  "resource_score":    <0-10>,
  "is_acceptable":     <true/false>,
  "key_issues":        ["issue1", "issue2"],
  "reasoning":         "brief explanation"
}}"""

    try:
        response = anthropic_client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=400,
            temperature=0,
            messages=[{"role": "user", "content": prompt}]
        )
        text = response.content[0].text.strip()
        if '```' in text:
            text = text.split('```')[1].split('```')[0]
            if text.startswith('json'):
                text = text[4:]
        return json.loads(text.strip())
    except Exception as e:
        print(f"  LLM judge failed: {e}")
        return None


# ── Master evaluation function ────────────────────────────────────────
def evaluate_generated_policy(query, generated_policy_str,
                               gt_policy_doc, run_llm_judge=True):
    """
    Run all four evaluation layers on a generated policy string.
    Returns a results dict with all metrics.
    """
    results = {"query": query}

    # Layer 1 — Syntactic validity
    is_valid, msg, policy_doc = check_layer1_syntactic(generated_policy_str)
    results["layer1_valid"]   = is_valid
    results["layer1_message"] = msg

    if not is_valid:
        results["layer2_hallucination_rate"] = None
        results["layer3_precision"]          = None
        results["layer3_recall"]             = None
        results["layer3_f1"]                 = None
        results["parse_failed"]              = True
        return results

    results["parse_failed"] = False

    # Layer 2 — Hallucination check
    l2 = check_layer2_hallucination(policy_doc)
    results["layer2_n_generated"]        = l2["n_generated"]
    results["layer2_n_hallucinated"]     = l2["n_hallucinated"]
    results["layer2_hallucination_rate"] = l2["hallucination_rate"]
    results["layer2_hallucinated"]       = l2["hallucinated"]

    # Layer 3 — Semantic action matching
    l3 = check_layer3_semantic(policy_doc, gt_policy_doc)
    results["layer3_precision"]                   = l3["precision"]
    results["layer3_recall"]                      = l3["recall"]
    results["layer3_f1"]                          = l3["f1"]
    results["layer3_exact_match"]                 = l3["exact_match"]
    results["layer3_covered"]                     = l3["covered"]
    results["layer3_missing"]                     = l3["missing"]
    results["layer3_extra"]                       = l3["extra"]
    results["layer3_n_generated"]                 = l3["n_generated"]
    results["layer3_n_expected"]                  = l3["n_expected"]
    results["layer3_resource_score"]              = l3["resource_score"]
    results["layer3_resource_mismatches"]         = l3["resource_mismatches"]
    results["layer3_gt_has_conditions"]           = l3["gt_has_conditions"]
    results["layer3_gen_has_conditions"]          = l3["gen_has_conditions"]
    results["layer3_condition_present_when_expected"] = l3["condition_present_when_expected"]
    results["layer3_has_deny"]                    = l3["has_deny"]

    # Layer 4 — LLM judge
    if run_llm_judge:
        l4 = check_layer4_llm_judge(query, policy_doc, gt_policy_doc, l3)
        if l4:
            results["layer4_functional_score"] = l4.get("functional_score")
            results["layer4_security_score"]   = l4.get("security_score")
            results["layer4_resource_score"]   = l4.get("resource_score")
            results["layer4_is_acceptable"]    = l4.get("is_acceptable")
            results["layer4_key_issues"]       = l4.get("key_issues", [])
            results["layer4_reasoning"]        = l4.get("reasoning", "")
        else:
            results["layer4_functional_score"] = None
            results["layer4_is_acceptable"]    = None

    return results


print("✓ All four evaluation layers defined")
print("  Layer 1: check_layer1_syntactic")
print("  Layer 2: check_layer2_hallucination")
print("  Layer 3: check_layer3_semantic")
print("  Layer 4: check_layer4_llm_judge")
print("  Master:  evaluate_generated_policy")

✓ GT database loaded: 4483 valid actions
✓ All four evaluation layers defined
  Layer 1: check_layer1_syntactic
  Layer 2: check_layer2_hallucination
  Layer 3: check_layer3_semantic
  Layer 4: check_layer4_llm_judge
  Master:  evaluate_generated_policy


In [9]:

def run_evaluation(run_llm_judge=True, sample_size=None):
    """
    Run full evaluation comparing RAG vs No RAG on all ground truth policies.

    Args:
        run_llm_judge: whether to run Layer 4 (costs GPT-4o API calls)
        sample_size:   if set, only evaluate first N policies (for testing)
    """
    print(f"Loading ground truth from: {GT_PATH}")
    with open(GT_PATH) as f:
        ground_truth = json.load(f)

    if sample_size:
        ground_truth = ground_truth[:sample_size]
        print(f"Sampling first {sample_size} policies for testing")

    print(f"Total policies to evaluate: {len(ground_truth)}")
    print(f"LLM judge enabled: {run_llm_judge}")
    print()

    results_log = []

    # Accumulators for summary
    rag_totals    = {"precision": 0, "recall": 0, "f1": 0,
                     "hall_rate": 0, "exact": 0,
                     "resource_score": 0, "acceptable": 0, "n": 0}
    norag_totals  = {"precision": 0, "recall": 0, "f1": 0,
                     "hall_rate": 0, "exact": 0,
                     "resource_score": 0, "acceptable": 0, "n": 0}

    for i, entry in enumerate(tqdm(ground_truth, desc="Evaluating")):
        query          = entry["query"]
        gt_policy_doc  = entry["policy_document"]
        policy_name    = entry["policy_name"]
        source         = entry["source"]

        print(f"\n[{i+1:02d}/{len(ground_truth)}] {policy_name} ({source})")
        print(f"  Query: {query[:80]}...")

        entry_result = {
            "policy_name": policy_name,
            "source":      source,
            "query":       query,
        }

        # ── No RAG generation ─────────────────────────────────────────
        try:
            no_rag_str    = generate_policy_no_rag(query)
            no_rag_result = evaluate_generated_policy(
                query, no_rag_str, gt_policy_doc,
                run_llm_judge=run_llm_judge
            )
            entry_result["no_rag"] = no_rag_result
            entry_result["no_rag"]["generated_policy_str"] = no_rag_str

            # Always count this attempt for the overall average
            norag_totals["n"] += 1

            if no_rag_result.get("parse_failed"):
                print(f"  No RAG  → Policy failed syntactic check (Layer 1): {no_rag_result.get('layer1_message')}")
                # Penalize metrics for parsing failure
                norag_totals["precision"]      += 0
                norag_totals["recall"]         += 0
                norag_totals["f1"]             += 0
                norag_totals["hall_rate"]      += 1.0 # 100% hallucination for invalid policy
                norag_totals["resource_score"] += 0
                norag_totals["exact"]          += 0
                if run_llm_judge:
                    norag_totals["acceptable"] += 0
            else:
                norag_totals["precision"]      += no_rag_result["layer3_precision"]
                norag_totals["recall"]         += no_rag_result["layer3_recall"]
                norag_totals["f1"]             += no_rag_result["layer3_f1"]
                norag_totals["hall_rate"]      += no_rag_result["layer2_hallucination_rate"]
                norag_totals["resource_score"] += no_rag_result["layer3_resource_score"]
                norag_totals["exact"]          += int(no_rag_result["layer3_exact_match"])
                if run_llm_judge and no_rag_result.get("layer4_is_acceptable") is not None:
                    norag_totals["acceptable"] += int(no_rag_result["layer4_is_acceptable"])

            # Safely get formatted values for printing
            no_rag_p_str   = f"{no_rag_result.get('layer3_precision'):.3f}" if no_rag_result.get('layer3_precision') is not None else "FAIL"
            no_rag_r_str   = f"{no_rag_result.get('layer3_recall'):.3f}" if no_rag_result.get('layer3_recall') is not None else "FAIL"
            no_rag_f1_str  = f"{no_rag_result.get('layer3_f1'):.3f}" if no_rag_result.get('layer3_f1') is not None else "FAIL"
            no_rag_hall_str= f"{no_rag_result.get('layer2_hallucination_rate'):.3f}" if no_rag_result.get('layer2_hallucination_rate') is not None else "FAIL"

            print(f"  No RAG  → P={no_rag_p_str} "
                  f"R={no_rag_r_str} "
                  f"F1={no_rag_f1_str} "
                  f"Hall={no_rag_hall_str}")

        except Exception as e:
            print(f"  No RAG failed: {e}")
            entry_result["no_rag"] = {"error": str(e)}
            # Penalize metrics for exceptions during generation/evaluation
            norag_totals["n"] += 1 # Count this attempt as well
            norag_totals["precision"]      += 0
            norag_totals["recall"]         += 0
            norag_totals["f1"]             += 0
            norag_totals["hall_rate"]      += 1.0
            norag_totals["resource_score"] += 0
            norag_totals["exact"]          += 0
            if run_llm_judge:
                norag_totals["acceptable"] += 0

        # ── RAG generation ────────────────────────────────────────────
        try:
            retrieved     = retrieve_for_generation_decomposed_v3(query)
            rag_str       = generate_policy_with_rag(query, retrieved)
            rag_result    = evaluate_generated_policy(
                query, rag_str, gt_policy_doc,
                run_llm_judge=run_llm_judge
            )
            entry_result["rag"] = rag_result
            entry_result["rag"]["generated_policy_str"] = rag_str
            entry_result["rag"]["n_chunks_retrieved"]   = len(retrieved)

            # Always count this attempt for the overall average
            rag_totals["n"] += 1

            if rag_result.get("parse_failed"):
                print(f"  RAG     → Policy failed syntactic check (Layer 1): {rag_result.get('layer1_message')}")
                # Penalize metrics for parsing failure
                rag_totals["precision"]      += 0
                rag_totals["recall"]         += 0
                rag_totals["f1"]             += 0
                rag_totals["hall_rate"]      += 1.0
                rag_totals["resource_score"] += 0
                rag_totals["exact"]          += 0
                if run_llm_judge:
                    rag_totals["acceptable"] += 0
            else:
                rag_totals["precision"]      += rag_result["layer3_precision"]
                rag_totals["recall"]         += rag_result["layer3_recall"]
                rag_totals["f1"]             += rag_result["layer3_f1"]
                rag_totals["hall_rate"]      += rag_result["layer2_hallucination_rate"]
                rag_totals["resource_score"] += rag_result["layer3_resource_score"]
                rag_totals["exact"]          += int(rag_result["layer3_exact_match"])
                if run_llm_judge and rag_result.get("layer4_is_acceptable") is not None:
                    rag_totals["acceptable"] += int(rag_result["layer4_is_acceptable"])

                # Safely get formatted values for printing
                rag_p_str   = f"{rag_result.get('layer3_precision'):.3f}" if rag_result.get('layer3_precision') is not None else "FAIL"
                rag_r_str   = f"{rag_result.get('layer3_recall'):.3f}" if rag_result.get('layer3_recall') is not None else "FAIL"
                rag_f1_str  = f"{rag_result.get('layer3_f1'):.3f}" if rag_result.get('layer3_f1') is not None else "FAIL"
                rag_hall_str= f"{rag_result.get('layer2_hallucination_rate'):.3f}" if rag_result.get('layer2_hallucination_rate') is not None else "FAIL"

                print(f"  RAG     → P={rag_p_str} "
                      f"R={rag_r_str} "
                      f"F1={rag_f1_str} "
                      f"Hall={rag_hall_str}")

        except Exception as e:
            print(f"  RAG failed: {e}")
            entry_result["rag"] = {"error": str(e)}
            # Penalize metrics for exceptions during generation/evaluation
            rag_totals["n"] += 1 # Count this attempt as well
            rag_totals["precision"]      += 0
            rag_totals["recall"]         += 0
            rag_totals["f1"]             += 0
            rag_totals["hall_rate"]      += 1.0
            rag_totals["resource_score"] += 0
            rag_totals["exact"]          += 0
            if run_llm_judge:
                rag_totals["acceptable"] += 0

        results_log.append(entry_result)

        # Save checkpoint every 10 policies
        if (i + 1) % 10 == 0:
            with open(RESULTS_PATH, "w") as f:
                json.dump(results_log, f, indent=2)
            print(f"  [checkpoint saved at {i+1} policies]")

    # ── Final save ────────────────────────────────────────────────────
    with open(RESULTS_PATH, "w") as f:
        json.dump(results_log, f, indent=2)

    # ── Summary table ─────────────────────────────────────────────────
    n_rag   = rag_totals["n"]
    n_norag = norag_totals["n"]

    print(f"\n{'='*65}")
    print(f"FINAL EVALUATION RESULTS")
    print(f"{'='*65}")
    print(f"{'Metric':<30} {'No RAG':>12} {'RAG':>12} {'Δ':>8}")
    print(f"{'─'*65}")

    metrics = [
        ("Avg Precision (L3)",    "precision"),
        ("Avg Recall (L3)",       "recall"),
        ("Avg F1 (L3)",           "f1"),
        ("Avg Hallucination (L2)","hall_rate"),
        ("Avg Resource Score (L3)","resource_score"),
        ("Exact Match Rate",     "exact"),
    ]

    for label, key in metrics:
        norag_val = norag_totals[key] / n_norag if n_norag else 0
        rag_val   = rag_totals[key]   / n_rag   if n_rag   else 0
        delta     = rag_val - norag_val
        # Hallucination lower is better — flip delta sign for display
        delta_str = f"{'+' if delta >= 0 else ''}{delta:.3f}"
        print(f"  {label:<28} {norag_val:>12.3f} {rag_val:>12.3f} {delta_str:>8}")

    if run_llm_judge:
        norag_acc = norag_totals["acceptable"] / n_norag if n_norag else 0
        rag_acc   = rag_totals["acceptable"]   / n_rag   if n_rag   else 0
        delta_acc = rag_acc - norag_acc
        print(f"  {'LLM Acceptable Rate (L4)':<28} "
              f"{norag_acc:>12.3f} {rag_acc:>12.3f} "
              f"{'+' if delta_acc >= 0 else ''}{delta_acc:.3f}")

    print(f"{'─'*65}")
    print(f"  Policies evaluated: No RAG={n_norag}, RAG={n_rag}")
    print(f"  Full results saved to: {RESULTS_PATH}")

    return results_log


In [11]:
#Run evaluation

# Quick test on 5 policies first to verify everything works
#results = run_evaluation(run_llm_judge=True, sample_size=10)

# Full evaluation with LLM judge (expensive — runs on all 75 policies)
results = run_evaluation(run_llm_judge=True)

Loading ground truth from: /content/drive/MyDrive/iam_rag_project/data/evaluation/evaluation_ground_truth.json
Total policies to evaluate: 75
LLM judge enabled: True



Evaluating:   0%|          | 0/75 [00:00<?, ?it/s]


[01/75] CodeBuildServiceRole (github)
  Query: Allow a build process to authenticate and manage container images in a private r...
  No RAG  → P=0.923 R=0.857 F1=0.889 Hall=0.000
  GPT-4o decomposed into 4 sub-queries:
    [explicit]   ecr: ECR describe repositories list images batch check layer availability put image get download url for layer
               reason: needs to manage container images including checking layer av
    [explicit]   s3: S3 get object put object list bucket
               reason: needs to retrieve and store objects in a storage service for
    [explicit]   logs: CloudWatch Logs create log group create log stream put log events
               reason: needs to write execution logs for monitoring purposes
    [implicit]   ecr: ECR get authorization token
               reason: always needed: get authorization token for ECR image pulling
  Total retrieved: 14 chunks (12 actions + 2 policies)


Evaluating:   1%|▏         | 1/75 [00:18<22:43, 18.42s/it]

  RAG     → P=0.500 R=0.714 F1=0.588 Hall=0.000

[02/75] CodePipelineServiceRole (github)
  Query: Allow a deployment pipeline to initiate and monitor build processes, manage and ...
  No RAG  → P=0.241 R=0.538 F1=0.333 Hall=0.172
  GPT-4o decomposed into 5 sub-queries:
    [explicit]   codepipeline: start pipeline execution get pipeline get pipeline execution
               reason: needs to initiate and monitor deployment pipeline
    [explicit]   codebuild: start build batch get build batch list build batches
               reason: needs to initiate and monitor build processes
    [explicit]   ecs: update service register task definition describe services describe task definition pass role iam passrole
               reason: needs to manage and update container task definitions and se
    [explicit]   s3: get object put object list bucket
               reason: needs to retrieve and store objects in a storage bucket
    [implicit]   sts: assume role
               reason: always need

...

  LLM judge failed: Unterminated string starting at: line 11 column 16 (char 813)
  RAG     → P=0.600 R=0.562 F1=0.581 Hall=0.067

[74/75] AWSStepFunctionsReadOnlyAccess (aws_managed)
  Query: Allow read-only access to view and list details of workflows, activities, execut...
  No RAG  → Policy failed syntactic check (Layer 1): Invalid JSON: Unterminated string starting at: line 128 column 9 (char 10236)
  No RAG  → P=FAIL R=FAIL F1=FAIL Hall=FAIL
  GPT-4o decomposed into 1 sub-queries:
    [explicit]   swf: list domains describe domain list workflows describe workflow executions list activity types describe activity type
               reason: needs list and describe actions for workflows, activities, e
  Total retrieved: 3 chunks (3 actions + 0 policies)


Evaluating:  99%|█████████▊| 74/75 [22:17<00:21, 21.55s/it]

  RAG     → Policy failed syntactic check (Layer 1): Invalid JSON: Expecting value: line 1 column 1 (char 0)

[75/75] AWSGlueSchemaRegistryReadonlyAccess (aws_managed)
  Query: Allow read-only access to view and query schema registries, schemas, and schema ...
  No RAG  → P=0.188 R=0.273 F1=0.222 Hall=0.688
  GPT-4o decomposed into 1 sub-queries:
    [explicit]   glue: get schema get schema version check schema version validity get tags
               reason: needs get schema, get schema version, check schema version v
  Total retrieved: 1 chunks (0 actions + 1 policies)


Evaluating: 100%|██████████| 75/75 [22:53<00:00, 18.32s/it]

  RAG     → P=0.000 R=0.000 F1=0.000 Hall=1.000

FINAL EVALUATION RESULTS
Metric                               No RAG          RAG        Δ
─────────────────────────────────────────────────────────────────
  Avg Precision (L3)                  0.499        0.486   -0.013
  Avg Recall (L3)                     0.600        0.688   +0.089
  Avg F1 (L3)                         0.528        0.548   +0.020
  Avg Hallucination (L2)              0.255        0.048   -0.207
  Avg Resource Score (L3)             0.827        0.973   +0.147
  Exact Match Rate                    0.120        0.053   -0.067
  LLM Acceptable Rate (L4)            0.147        0.040 -0.107
─────────────────────────────────────────────────────────────────
  Policies evaluated: No RAG=75, RAG=75
  Full results saved to: /content/drive/MyDrive/iam_rag_project/results/evaluation_results.json
